In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
#import torch
import anndata as ad
import scanpy as sc
import polars as pl
import seaborn as sns

from scanpy.plotting import palettes
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

In [ ]:
color_map = {
    'autophagy inducer': (0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
    'apoptosis': (1.0, 0.4980392156862745, 0.054901960784313725),
    'ferroptosis inducer': (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
    'immunogenic cell death': (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
    'pyroptosis inducer': (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
    'necroptosis inducer': (0.5490196078431373, 0.33725490196078434, 0.29411764705882354)
}

# Either load in single-cell profiles and create adata and run UMAP or directly load in adata embedding files!

In [ ]:
def run_scanpy(adata):
    print("Starting scanpy!")
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=500, n_pcs=50)
    sc.tl.paga(adata, groups="Metadata_cmpdName")
    sc.pl.paga(adata, plot=False)  # remove `plot=False` if you want to see the coarse-grained graph
    sc.tl.umap(adata, init_pos='random')
    print("Embedding complete. Saving file!")

In [ ]:
cp_sc_profiles = pl.read_parquet("singlecell_features_CellProfiler.parquet") #replace with according path

In [ ]:
meta_cols_cp = [col for col in cp_sc_profiles.columns if "Metadata" in col] + ["moa", "label"]

In [ ]:
adata_CP = ad.AnnData(X = cp_sc_profiles[[col for col in cp_sc_profiles.columns if col not in meta_cols_cp]].to_pandas(), obs = cp_sc_profiles[[col for col in cp_sc_profiles.columns if col in meta_cols_cp]].to_pandas())

In [ ]:
dp_sc_profiles = pl.read_parquet("singlecell_features_DeepProfiler.parquet") #replace with according path
meta_cols_dp = [col for col in dp_sc_profiles.columns if "Metadata" in col] + ["moa", "label"]
adata_DP = ad.AnnData(X = dp_sc_profiles[[col for col in dp_sc_profiles.columns if col not in meta_cols_dp]].to_pandas(), obs = dp_sc_profiles[[col for col in dp_sc_profiles.columns if col in meta_cols_dp]].to_pandas())

In [ ]:
dino_sc_profiles = pl.read_parquet("singlecell_features_DINO.parquet") #replace with according path
meta_cols_dino = [col for col in dino_sc_profiles.columns if "Metadata" in col] + ["moa", "label"]
adata_DIBO = ad.AnnData(X = dino_sc_profiles[[col for col in dino_sc_profiles.columns if col not in meta_cols_dino]].to_pandas(), obs = dino_sc_profiles[[col for col in dino_sc_profiles.columns if col in meta_cols_dino]].to_pandas())

In [ ]:
adata_DINO = ad.read_h5ad("/share/data/analyses/benjamin/DINO/CELLDEATH/scripts/paper_code/data_revision/embeddings/sc_embedding_DINO.h5ad")
adata_DP = ad.read_h5ad("/share/data/analyses/benjamin/DINO/CELLDEATH/scripts/paper_code/data_revision/embeddings/sc_embedding_DP.h5ad")
adata_CP = ad.read_h5ad("/share/data/analyses/benjamin/DINO/CELLDEATH/scripts/paper_code/data_revision/embeddings/sc_embedding_CP.h5ad")

In [ ]:
adata_DINO = adata_DINO[adata_DINO.obs["outliers"] == "unassigned"] #drop outliers

# Dataset overview (cell counts, viability, qc)

## Plotting helper fct. for overview

In [ ]:
def plot_plate_heatmap_sites(
    df, 
    value_col='Value', 
    well_col='Metadata_Well', 
    site_col='Metadata_Site', 
    Barcode=None,
    ax=None  # NEW PARAMETER
):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    if Barcode is not None:
        df = df[df['Metadata_Plate'] == Barcode]

    plate_rows, plate_cols, subgrid_size = 16, 24, 3
    total_rows, total_cols = plate_rows * subgrid_size, plate_cols * subgrid_size

    if pd.api.types.is_numeric_dtype(df[value_col]):
        is_numeric = True
        grid = np.full((total_rows, total_cols), np.nan, dtype=float)
    else:
        is_numeric = False
        grid = np.full((total_rows, total_cols), np.nan, dtype=float)
        categories = sorted(df[value_col].dropna().unique())
        cat_to_code = {cat: code for code, cat in enumerate(categories)}
        code_to_cat = {code: cat for cat, code in cat_to_code.items()}

    for _, row in df.iterrows():
        well = row[well_col]
        site = int(row[site_col])
        value = row[value_col]

        well_row_letter = well[0]
        well_col_num = int(well[1:])
        well_row_index = ord(well_row_letter.upper()) - ord('A')
        well_col_index = well_col_num - 1

        sub_row = (site - 1) // subgrid_size
        sub_col = (site - 1) % subgrid_size
        global_row = well_row_index * subgrid_size + sub_row
        global_col = well_col_index * subgrid_size + sub_col

        if is_numeric:
            grid[global_row, global_col] = value
        else:
            grid[global_row, global_col] = cat_to_code[value]

    # Use provided ax or create new one
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 10))
    else:
        fig = ax.figure  # grab the parent figure

    if is_numeric:
        cmap = 'viridis'
        cax = ax.imshow(grid, interpolation='nearest', cmap=cmap,
                        extent=[0, total_cols, 0, total_rows], origin='upper',vmin=0, vmax=50)
        fig.colorbar(cax, ax=ax)
    else:
        num_categories = len(cat_to_code)
        cmap = plt.get_cmap('tab20', num_categories)
        cax = ax.imshow(grid, interpolation='nearest', cmap=cmap,
                        extent=[0, total_cols, 0, total_rows], origin='upper')
        cbar = fig.colorbar(cax, ax=ax, ticks=np.arange(num_categories))
        cbar.ax.set_yticklabels([code_to_cat[i] for i in range(num_categories)])
        cbar.set_label(value_col)

    ax.set_title(f'Heatmap of {value_col} for Plate {Barcode}')

    x_major_ticks = np.arange(plate_cols) * subgrid_size + subgrid_size / 2
    y_major_ticks = np.arange(plate_rows) * subgrid_size + subgrid_size / 2
    x_labels = [f"{col+1:02d}" for col in range(plate_cols)]
    y_labels = [chr(ord('A') + i) for i in range(plate_rows)][::-1]

    ax.set_xticks(x_major_ticks)
    ax.set_yticks(y_major_ticks)
    ax.set_xticklabels(x_labels)
    ax.set_yticklabels(y_labels)

    x_minor_ticks = np.arange(0, total_cols + 1, subgrid_size)
    y_minor_ticks = np.arange(0, total_rows + 1, subgrid_size)
    ax.set_xticks(x_minor_ticks, minor=True)
    ax.set_yticks(y_minor_ticks, minor=True)

    ax.set_axisbelow(False)
    ax.grid(which='minor', color='white', linestyle='-', linewidth=1)
    plt.setp(ax.get_xticklabels(), ha="center", rotation_mode="anchor")

    if ax is None:
        plt.tight_layout()
        plt.show()

def plot_dose_response_per_moa(df, class_column="Timepoint", response_metric="viability", moa_column="moa", save_dir=None):
    sns.set(style="white", context="talk")
    #df = df[df["Metadata_cmpdConc"] != 0]
    for moa in df[moa_column].dropna().unique():
        df_moa = df[df[moa_column] == moa]
        compounds = sorted(df_moa["Metadata_cmpdName"].dropna().astype(str).unique())
        n = len(compounds)
        ncols = 4
        nrows = -(-n // ncols)

        fig_width = ncols * 3
        fig_height = nrows * 3
        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(fig_width, fig_height),
            sharex=False, sharey=True,
            constrained_layout=True
        )
        axes = axes.flatten()
        
        for i in range(ncols * nrows):
            ax = axes[i]
            if i < len(compounds):
                compound = compounds[i]
                data = df_moa[df_moa["Metadata_cmpdName"] == compound]
                grouped = data.groupby(["Metadata_cmpdConc", class_column])[response_metric].agg(["mean", "sem"]).reset_index()

                for cls, subdf in grouped.groupby(class_column):
                    #x_vals = np.log10(subdf["Metadata_cmpdConc"])
                    x_vals = subdf["Metadata_cmpdConc"]
                    y_vals = subdf["mean"]
                    y_errs = subdf["sem"]

                    if y_vals.isnull().all() or y_errs.isnull().all():
                        continue

                    ax.errorbar(
                        x=x_vals,
                        y=y_vals,
                        yerr=y_errs,
                        label=str(cls),
                        fmt='-o',
                        capsize=2,
                        color="black",
                        markersize=4,
                        linewidth=1
                    )

                ax.set_title(compound, fontweight='bold', fontsize=10)

                if i % ncols == 0:
                    ax.set_ylabel("Viability (%)", fontsize=9)
                else:
                    ax.set_ylabel("")

                ax.set_xlabel("Concentration (µM)", fontsize=9)
                #else:
                #    ax.set_xlabel("")

                ax.tick_params(
                    axis='both', which='both', direction='out', length=5, width=1.0,
                    top=False, right=False, bottom=True, left=True, labelsize=8
                )
                xticks = np.arange(0, 11, 2)
                ax.set_xticks(xticks)
                ax.set_xticklabels([str(tick) for tick in xticks])
                ax.set_yticks(np.linspace(0, 1.5, 6))
                ax.set_box_aspect(1)
                ax.spines[['top', 'right']].set_visible(False)
            else:
                ax.set_visible(False)

        fig.suptitle(f"MOA: {moa}", fontsize=13, fontweight='bold')
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            fname = f"{moa.replace('/', '_').replace(' ', '_')}_dose_response.png"
            fig.savefig(os.path.join(save_dir, fname), dpi=300)

        plt.show()

def plot_binned_compound_concentration_barplots_polars(df_pl, compound_col="Metadata_cmpdName", conc_col="Metadata_cmpdConc",
                                                        expected_concs=[0.1, 0.3, 1.0, 3.0, 5.0, 10.0],
                                                        palette="viridis",
                                                        figsize=(22, 8)):
    import math

    # Round concentrations
    df_pl = df_pl.with_columns(
        pl.col(conc_col).cast(pl.Float64).map_elements(
            lambda x: min(expected_concs, key=lambda y: abs(y - x))
        ).alias(conc_col)
    )

    # Group and count
    grouped = df_pl.group_by([compound_col, conc_col]).agg(
        pl.count().alias("count")
    ).sort([compound_col, conc_col])

    # Compute max per compound to bin them
    max_counts = grouped.group_by(compound_col).agg(pl.max("count").alias("max_count"))
    expr = pl.when(pl.col("max_count") < 10000).then(pl.lit("<10k cells"))
    expr = expr.when(pl.col("max_count") <= 40000).then(pl.lit("10k–40k cells"))
    expr = expr.otherwise(pl.lit( ">40k cells"))

    max_counts = max_counts.with_columns(
        expr.alias("bin")
    )

    # Merge bin info back into grouped
    grouped = grouped.join(max_counts.select([compound_col, "bin"]), on=compound_col, how="left")

    # Convert to pandas for seaborn
    df = grouped.to_pandas()
    df[conc_col] = pd.Categorical(df[conc_col], categories=sorted(expected_concs), ordered=True)
    bin_labels = ["<10k cells", "10k–40k cells", ">40k cells"]
    bins_present = [b for b in bin_labels if b in df["bin"].unique()]
    
    fig, axes = plt.subplots(
        len(bins_present),
        1,
        figsize=(figsize[0], figsize[1] * len(bins_present)),
        dpi=300
    )

    if len(bins_present) == 1:
        axes = [axes]

    for i, bin_label in enumerate(bins_present):
        bin_df = df[df["bin"] == bin_label]
        compounds = sorted(bin_df[compound_col].unique())

        sns.barplot(
            data=bin_df,
            x=compound_col,
            y="count",
            hue=conc_col,
            hue_order=sorted(expected_concs),
            order=compounds,
            ax=axes[i],
            palette=palette
        )

        axes[i].set_title(f"{bin_label} (n={len(compounds)} compounds)", fontsize=13, fontweight="bold")
        axes[i].set_ylabel("Cell Count", fontweight="bold", fontsize=14)
        axes[i].tick_params(axis='x', labelrotation=90)

        # Show x-axis label only on bottom plot
        if i == len(axes) - 1:
            axes[i].set_xlabel("Compound", fontweight="bold", fontsize=14)
        else:
            axes[i].set_xlabel("")

        # Remove legend
        axes[i].get_legend().remove()

    plt.suptitle("Compound-wise Cell Counts by Concentration (Binned by Max Count)", fontsize=16, fontweight='bold')

    # Reduce vertical spacing
    fig.subplots_adjust(hspace=0.55)

    # Save to file
    #plt.savefig("compound_concentration_barplot.png", bbox_inches="tight", dpi=300)
    plt.show()

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_obs_count_bargrid(
    df,
    compound_col="Metadata_cmpdName",
    conc_col="Metadata_cmpdConc",
    class_col="moa",
    expected_concs=[0.1, 0.3, 1.0, 3.0, 5.0, 10.0],
    figsize=(14, 12),
    color_map=None,
    ymax=None
):
    """
    Barplot grid showing counts per compound, split by concentration,
    with color-coded bars based on a class column (e.g., 'moa').
    """

    # Round concentration values to nearest expected level
    df[conc_col] = df[conc_col].astype(float).apply(lambda x: min(expected_concs, key=lambda y: abs(y - x)))
    df[conc_col] = pd.Categorical(df[conc_col], categories=sorted(expected_concs), ordered=True)

    # Count per compound + concentration and get class
    count_df = df.groupby([compound_col, conc_col, class_col]).size().reset_index(name='count')

    # Ensure compound order is consistent
    compound_order = sorted(count_df[compound_col].unique())

    # Custom plotting function
    def colored_barplot(data, x, y, ax=None, order=None, **kwargs):
        if ax is None:
            ax = plt.gca()

        for i, row in data.iterrows():
            xpos = order.index(row[compound_col])
            color = color_map.get(row[class_col], "gray")
            ax.bar(x=xpos, height=row[y], color=color, width=0.8)

            # Annotate if height > ymax
            if ymax is not None and row[y] > ymax:
                ax.annotate(f"{int(row[y])}",
                            xy=(xpos, ymax),
                            xytext=(0, 3),
                            textcoords='offset points',
                            ha='center', va='bottom',
                            fontsize=10, fontweight='bold')

        # Fix x-axis ticks
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=90 if kwargs.get("rotate_xticks", False) else 0)

        if ymax is not None:
            ax.set_ylim(0, ymax)

    # Create FacetGrid
    g = sns.FacetGrid(
        data=count_df,
        row=conc_col,
        row_order=sorted(expected_concs),
        height=2.5,
        aspect=figsize[0] / figsize[1],
        sharex=True,
        sharey=False
    )

    # Map colored barplot
    g.map_dataframe(
        colored_barplot,
        x=compound_col,
        y="count",
        order=compound_order,
        rotate_xticks=True
    )

    # Set labels and titles
    g.set_axis_labels("Compound", "Cell Count")
    g.set_titles(row_template="{row_name} µM", fontweight='bold')

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_dose_response_grid(
    df,
    compound_list,
    ncols=4,
    nrows=3,
    class_column="moa",
    response_metric="viability",
    save_path=None
):
    import os
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt

    sns.set(style="white", context="talk")
    #df = df[df["Metadata_cmpdConc"] != 0]  # Exclude 0 concentration points

    fig_width = ncols * 3
    fig_height = nrows * 3
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(fig_width, fig_height),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )
    axes = axes.flatten()

    for i in range(ncols * nrows):
        ax = axes[i]
        if i < len(compound_list):
            compound = compound_list[i]
            data = df[df["Metadata_cmpdName"] == compound]
            if data.empty:
                ax.set_title(f"{compound}\n(no data)", fontsize=9, fontstyle="italic")
                ax.set_visible(True)
                continue

            grouped = data.groupby(["Metadata_cmpdConc", class_column])[response_metric].agg(["mean", "sem"]).reset_index()

            for cls, subdf in grouped.groupby(class_column):
                #x_vals = np.log10(subdf["Metadata_cmpdConc"].replace(0, np.nan))
                x_vals = subdf["Metadata_cmpdConc"]
                y_vals = subdf["mean"]
                y_errs = subdf["sem"]

                if y_vals.isnull().all() or y_errs.isnull().all():
                    continue

                ax.errorbar(
                    x=x_vals,
                    y=y_vals,
                    yerr=y_errs,
                    label=str(cls),
                    fmt='-o',
                    capsize=2,
                    color="black",
                    markersize=4,
                    linewidth=1
                )

            ax.set_title(compound, fontweight='bold', fontsize=15)

            ax.set_xticks(np.arange(0, 11, 2))  # typical log10 concentration range
            ax.set_yticks(np.linspace(0, 1.5, 6))  # 0 to 1.5 in 0.3 steps

            ax.tick_params(
                axis='both', which='both',
                direction='out',
                length=4,
                width=1.0,
                labelsize=8,
                top=False, right=False, bottom=True, left=True
            )

            ax.set_box_aspect(1)
            ax.spines[['top', 'right']].set_visible(False)

            if i % ncols == 0:
                ax.set_ylabel("Viability (%)", fontsize=12)
            else:
                ax.set_ylabel("")

            if i >= (nrows - 1) * ncols:
                ax.set_xlabel("", fontsize=12)
            else:
                ax.set_xlabel("")
        else:
            ax.set_visible(False)

    if save_path:
        fig.savefig(save_path, dpi=300)

    plt.show()


In [ ]:
qc_df = pd.read_csv("data_revision/qc_df.csv")
qc_df["site_int"] = qc_df["Metadata_Site"].str.extract(r"(\d+)").astype(int)
qc_df = qc_df[qc_df["Metadata_cmpdName"].isin(['2;3-DCPE HYDROCHLORIDE',
 'ANATOXIN A FUMARATE',
 'APOPTOSIS ACTIVATOR 2',
 'AT 101',
 'AZD 2461',
 'BETULINIC ACID',
 'BLEOMYCIN',
 'BORTEZOMIB',
 'BZ 423',
 'C 75',
 'CAMPTOTHECIN',
 'CARBOPLATIN',
 'CISPLATIN',
 'CLADRIBINE',
 'CRIZOTINIB',
 'DACTINOMYCIN (ACTINOMYCIN D)',
 'DAUNORUBICIN',
 'DOMPERIDONE',
 'DOXORUBICIN',
 'EPIRUBICIN',
 'ERASTIN',
 'ETOPOSIDE',
 'EVEROLIMUS',
 'FIN 56',
 'FK 866',
 'FLUDARABINE',
 'G5',
 'HS-173',
 'KAEMPFEROL',
 'L-690;330',
 'LOPERAMIDE',
 'LOVASTATIN',
 'MITOMYCIN C',
 'MPC 6827',
 'NARCICLASINE',
 'NIGERICIN',
 'OLAPARIB',
 'ONCRASIN 1',
 'OXALIPLATIN',
 'PIMOZIDE',
 'POLYPHYLLIN VI',
 'PRAVASTATIN',
 'QUERCETIN',
 'RIFAXIMIN',
 'SHIKONIN',
 'SIMVASTATIN',
 'SMBA 1',
 'SN 38',
 'SORAFENIB',
 'SULFASALAZINE',
 'TENIPOSIDE',
 'TOPOTECAN',
 'VU0359595'])]

In [ ]:
selected_compounds = ["BZ 423", "EVEROLIMUS", "CRIZOTINIB", "SIMVASTATIN", "HS-173", "NIGERICIN"]
plot_dose_response_grid(df=qc_df, compound_list=selected_compounds, ncols=6, nrows=1, save_path ="fig1_paneld.png")

In [ ]:
 plot_dose_response_per_moa(qc_df, class_column="moa", response_metric="viability", moa_column="moa", save_dir="plots")

In [ ]:
plot_binned_compound_concentration_barplots_polars(
    df_pl=combined_df, #use a df that has one row for each cell here. Can also modify above plotting to use qc df and use Count_nuclei column 
    compound_col="Metadata_cmpdName",
    conc_col="Metadata_cmpdConc"
)

## Compare distribution of cells by moa

In [ ]:
plot_obs_count_bargrid(
    df= combined_df.to_pandas(),
    compound_col="Metadata_cmpdName",
    conc_col="Metadata_cmpdConc",  # e.g. concentration
    figsize=(35, 10),
    color_map = color_map
)

# UMAP analysis

In [ ]:
sc.pl.umap(adata_CP, color = ['moa'])
sc.pl.umap(adata_DP, color = ['moa'])
sc.pl.umap(adata_DINO, color = ['moa'])


## Run unsupervised analysis (only if not already loaded embeddings)

In [ ]:
run_scanpy(adata_DINO)
run_scanpy(adata_DP)
run_scanpy(adata_CP)

In [ ]:
sc.pl.pca(adata_CP, color = ['moa'])
sc.pl.pca(adata_DP, color = ['moa'])
sc.pl.pca(adata_DINO, color = ['moa'])

In [ ]:
sc.pl.umap(adata_DINO, color = ['moa'], title ="DINO")
sc.pl.umap(adata_DP, color = ['moa'], title ="DP")
sc.pl.umap(adata_CP,color = ['moa'], title ="CP")

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from scanpy.plotting import palettes
from matplotlib.lines import Line2D

def plot_combined_umap(
    adata1, adata2, adata3, color_column="Metadata_cmpdName", 
    title1="CellProfiler", title2="DeepProfiler", title3="DINO", 
    save_path=None
):
    """
    Plots UMAPs of three AnnData objects with a shared color map and a single legend.
    
    Parameters:
    - adata1, adata2, adata3: AnnData objects to plot.
    - color_column (str): Column in `.obs` to use for coloring.
    - title1, title2, title3 (str): Titles for each subplot.
    - save_path (str, optional): File path to save the figure. If None, it won’t save the figure.
    
    Returns:
    - None: Displays the combined plot with a single legend.
    """
    # Use the standard Scanpy color palette
    default_palette = palettes.default_102[:60]  # Support up to 60 colors from Scanpy
    assert len(default_palette) >= 60, "Scanpy palette does not support more than 60 colors."

    # Get all unique categories across the three AnnData objects
    unique_categories = (
        set(adata1.obs[color_column].unique())
        .union(adata2.obs[color_column].unique())
        .union(adata3.obs[color_column].unique())
    )
    
    # Map the colors to the unique categories and sort by category name
    palette = default_palette[:len(unique_categories)]  # Adjust palette size to unique categories
    category_colors = dict(zip(sorted(unique_categories), palette))  # Sort categories alphabetically

    # Handle missing categories in each AnnData object
    def assign_colors(adata, color_column, category_colors):
        adata_categories = adata.obs[color_column].astype("category").cat.categories
        adata.uns[f"{color_column}_colors"] = [
            category_colors[cat] if cat in category_colors else "#d3d3d3"  # Assign default gray for missing categories
            for cat in adata_categories
        ]

    assign_colors(adata1, color_column, category_colors)
    assign_colors(adata2, color_column, category_colors)
    assign_colors(adata3, color_column, category_colors)

    # Create a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot UMAP for each dataset
    sc.pl.umap(adata1, color=[color_column], title=title1, ax=axes[0], legend_loc="none", show=False,)
    sc.pl.umap(adata2, color=[color_column], title=title2, ax=axes[1], legend_loc="none", show=False)
    sc.pl.umap(adata3, color=[color_column], title=title3, ax=axes[2], legend_loc="none", show=False)

    # Create a single legend using alphabetically sorted categories and colors
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=category_colors[cat], markersize=10, label=cat)
        for cat in sorted(unique_categories)
    ]
    fig.legend(handles=legend_elements, title=color_column, loc="center right", bbox_to_anchor=(1.05, 0.5))

    # Adjust layout and optionally save the figure
    plt.tight_layout(rect=[0, 0, 0.9, 1])  # Leave space on the right for the legend
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
from sklearn.cluster import DBSCAN
from adjustText import adjust_text

def plot_combined_umap(
    adata1, adata2, adata3, color_column="Metadata_cmpdName", 
    title1="CellProfiler", title2="DeepProfiler", title3="DINO", 
    save_path=None,
    dbscan_eps=0.8,
    dbscan_min_samples=500,
    alpha=0.3
):
    """
    Plots UMAPs of three AnnData objects with a shared color map, a single legend,
    numeric labels for each connected subcluster per category, and non-overlapping annotations.

    Overlapping labels are resolved using adjustText.
    """
    # Prepare color palette
    default_palette = palettes.default_102[:60]
    unique_categories = (
        set(adata1.obs[color_column].unique())
        .union(adata2.obs[color_column].unique())
        .union(adata3.obs[color_column].unique())
    )
    unique_categories = unique_categories - {"CYCLOPHOSPHAMIDE"}
    sorted_cats = sorted(unique_categories)
    category_colors = dict(zip(sorted_cats, default_palette))

    # Assign colors to AnnData
    def assign_colors(adata):
        cats = adata.obs[color_column].astype('category').cat.categories
        adata.uns[f"{color_column}_colors"] = [category_colors.get(c, '#d3d3d3') for c in cats]
    for ad in (adata1, adata2, adata3):
        assign_colors(ad)

    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(25, 10))
    datasets = [(adata1, axes[0], title1), (adata2, axes[1], title2), (adata3, axes[2], title3)]

    for adata, ax, title in datasets:
        # Plot UMAP
        sc.pl.umap(
            adata,
            color=[color_column],
            ax=ax,
            show=False,
            alpha=alpha,
            title=None,
            legend_loc='none'
        )
        # Remove axes
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(''); ax.set_ylabel('')

        coords = adata.obsm['X_umap']
        texts = []
        # annotate subclusters per category
        for idx, cat in enumerate(sorted_cats, start=1):
            mask = adata.obs[color_column] == cat
            pts = coords[mask.values]
            if pts.shape[0] == 0:
                continue
            # cluster subclusters
            if pts.shape[0] >= dbscan_min_samples:
                clustering = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit(pts)
                labels = clustering.labels_
            else:
                labels = np.zeros(pts.shape[0], dtype=int)
            # annotate each subcluster
            for sublabel in np.unique(labels):
                subpts = pts[labels == sublabel]
                centroid = subpts.mean(axis=0)
                dists = np.linalg.norm(subpts - centroid, axis=1)
                medoid = subpts[np.argmin(dists)]
                t = ax.text(
                    medoid[0], medoid[1], str(idx), fontsize=10,
                    color='black', ha='center', va='center', weight='bold', zorder=10
                )
                texts.append(t)
        # Adjust text to avoid overlaps
        adjust_text(
            texts, ax=ax,
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.7),
            expand_text=(1.2, 1.2), expand_points=(1.2, 1.2)
        )

    # Legend
    legend_elems = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=category_colors[cat], markersize=12,
               label=f"{i}. {cat}")
        for i, cat in enumerate(sorted_cats, start=1)
    ]
    #fig.legend(handles=legend_elems, title=color_column, loc='center right', bbox_to_anchor=(1.05, 0.5))
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_combined_umap(adata_DINO, adata_DP, adata_CP, color_column="Metadata_cmpdName")

# save_path="big_combined_umap.png"

In [ ]:
plot_combined_umap(adata_agg_CP, adata_agg_DP, adata_agg_DINO, color_column="Metadata_cmpdName") #get dfs from calling aggregated results further down


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import seaborn as sns
from matplotlib.lines import Line2D    
def make_jointplot_anndata(adata, dmso_name, colouring, cmpd, save_path=None):
    # Extract UMAP data from .obsm
    umap_data = pd.DataFrame(adata.obsm['X_umap'], columns=['UMAP1', 'UMAP2']).reset_index()

    # Join with metadata from .obs
    embedding = pd.concat([umap_data,pd.DataFrame(adata.obs).reset_index()], axis = 1)
    #embedding = embedding.reset_index()
    embedding[colouring] = embedding[colouring].str.strip('[]')
    #embedding["Metadata_cmpdNameConc2"] = embedding["Metadata_cmpdName"].astype(str) + "_" + embedding["Metadata_cmpdConc"].astype(str)
    # Generate a color palette based on unique values in the colouring column
    unique_treatments = embedding[colouring].unique()
    palette = sns.color_palette("tab10", len(unique_treatments))
    color_map = dict(zip(unique_treatments, palette))
    #print(color_map)
    # Adjust colors and transparency if colouring is 'Metadata_cmpdName'
    if colouring == 'Metadata_cmpdName':
        if dmso_name in color_map:
            color_map[dmso_name] = "lightgrey"
    color_map = {'autophagy inducer': (0.12156862745098039, 0.4666666666666667, 0.7058823529411765), 'apoptosis': (1.0, 0.4980392156862745, 0.054901960784313725), 'ferroptosis inducer': (0.17254901960784313, 0.6274509803921569, 0.17254901960784313), 'immunogenic cell death': (0.8392156862745098, 0.15294117647058825, 0.1568627450980392), 'pyroptosis inducer': (0.5803921568627451, 0.403921568627451, 0.7411764705882353), 'necroptosis inducer': (0.5490196078431373, 0.33725490196078434, 0.29411764705882354)}
    embedding['color'] = embedding[colouring].map(color_map)
    point_size = 10
    embedding['size'] = point_size
    
    # Increase the DPI for displaying
    plt.rcParams['figure.dpi'] = 300
    
    # Create the base joint plot
    g = sns.JointGrid(x='UMAP1', y='UMAP2', data=embedding, height=10)

    # Plot KDE plots for each category
    for treatment in sorted(unique_treatments):  # Sort alphabetically
        subset = embedding[embedding[colouring] == treatment]
        
        sns.kdeplot(x=subset["UMAP1"], ax=g.ax_marg_x, fill=True, color=color_map[treatment], legend=False)
        sns.kdeplot(y=subset["UMAP2"], ax=g.ax_marg_y, fill=True, color=color_map[treatment], legend=False)

    # Plot the scatter plots
    for treatment in sorted(unique_treatments):  # Sort alphabetically
        subset = embedding[embedding[colouring] == treatment]
        alpha_val = 0.3 if treatment == dmso_name and colouring == 'Metadata_cmpdName' else 0.8
        g.ax_joint.scatter(subset["UMAP1"], subset["UMAP2"], c=subset['color'], s=subset['size'], label=treatment, alpha=alpha_val, linewidth=0.0)
    
    g.ax_joint.set_title(cmpd)
    g.ax_joint.grid(False)
    g.ax_marg_x.grid(False)
    g.ax_marg_y.grid(False)
    #legend = g.ax_joint.legend(fontsize=10)
    #legend.get_frame().set_facecolor('white')
    #legend_elements = [Line2D([0], [0], marker='o', linestyle="None", color=color_map[treatment], label=f"{treatment}", markersize=5, markerfacecolor=color_map[treatment], alpha=1) for treatment in sorted(unique_treatments)]  # Sort alphabetically
    #legend = g.ax_joint.legend(handles=legend_elements, fontsize=10, title="moa")
    #legend.get_frame().set_facecolor('white')
    
    if save_path != None:
        current_time = datetime.datetime.now()
        timestamp = current_time.strftime("%Y%m%d_%H%M%S")
        g.savefig(f"{save_path}.png", dpi=300)

    plt.show()

In [ ]:
make_jointplot_anndata(adata_agg_DINO, 'DMSO', "moa", "") # first run cells in aggregated analysis below

In [ ]:
make_jointplot_anndata(adata_agg_DP, 'DMSO', "moa", "")

In [ ]:
make_jointplot_anndata(adata_agg_CP, 'DMSO', "moa", "")

## Density analysis 

In [ ]:
sc.tl.embedding_density(adata_DINO, basis='umap', groupby='moa')
sc.pl.embedding_density(adata_DINO, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_moa')

sc.tl.embedding_density(adata_CP, basis='umap', groupby='moa')
sc.pl.embedding_density(adata_CP, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_moa')

sc.tl.embedding_density(adata_DP, basis='umap', groupby='moa')
sc.pl.embedding_density(adata_DP, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_moa')

In [ ]:
adata_CP.obs["moa"] = adata_CP.obs["moa"].astype("category")

# 2. Sort categories alphabetically and apply the order
adata_CP.obs["moa"] = adata_CP.obs["moa"].cat.set_categories(
    sorted(adata_CP.obs["moa"].unique()), ordered=True
)

sc.tl.embedding_density(adata_CP, basis='umap', groupby='moa')
sc.pl.embedding_density(adata_CP, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_moa')

In [ ]:
sc.tl.embedding_density(adata_filter_grit, basis='umap', groupby='Metadata_cmpdName')
sc.pl.embedding_density(adata_filter_grit, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_Metadata_cmpdName')

## Per moa  

### CellProfiler

In [ ]:
#moas = list(adata_CP.obs["moa"].unique())
moas = ["apoptosis"]
adata_moa_cp = {}
for moa in moas:
    print(moa)
    filter_adata = adata_CP[adata_CP.obs["moa"] == moa]
    run_scanpy(filter_adata)
    sc.pl.umap(filter_adata, color = "Metadata_cmpdName")
    sc.tl.embedding_density(filter_adata, basis='umap', groupby='Metadata_cmpdName')
    sc.pl.embedding_density(filter_adata, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_Metadata_cmpdName')
    adata_moa_cp[moa] = filter_adata

### DeepProfiler

In [ ]:
#moas = list(adata_DP.obs["moa"].unique())
adata_moa_dp = {}
for moa in moas:
    print(moa)
    filter_adata = adata_DP[adata_DP.obs["moa"] == moa]
    run_scanpy(filter_adata)
    sc.pl.umap(filter_adata, color = "Metadata_cmpdName")
    sc.tl.embedding_density(filter_adata, basis='umap', groupby='Metadata_cmpdName')
    sc.pl.embedding_density(filter_adata, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_Metadata_cmpdName')
    adata_moa_dp[moa] = filter_adata

### DINO

In [ ]:
#moa = adata_filter_grit.obs["moa"].unique()
adata_moa_DINO = {}
for m in moas:
    print(m)
    adata_filt = adata_DINO[adata_DINO.obs["moa"]== m]
    run_scanpy(adata_filt)
    sc.pl.umap(adata_filt, color = "Metadata_cmpdName")
    sc.tl.embedding_density(adata_filt, basis='umap', groupby='Metadata_cmpdName')
    sc.pl.embedding_density(adata_filt, bg_dotsize = 40, fg_dotsize= 150, basis='umap', key='umap_density_Metadata_cmpdName')
    adata_moa_DINO[moa] = adata_filt

## Apoptosis analysis

In [ ]:
adata_apoptosis_DINO = #use the dictionary item for apoptosis from the previous call above
apoptosis_adata_DP = #use the dictionary item for apoptosis from the previous call above
apoptosis_adata_CP = #use the dictionary item for apoptosis from the previous call above

In [ ]:
def plot_combined_umap_apoptosis(adata1, adata2, adata3, color_column="Metadata_cmpdName", title1="DINO", title2="DeepProfiler", title3="CellProfiler", save_path=None):
    """
    Plots UMAPs of three AnnData objects with a shared color map and a single legend.
    
    Parameters:
    - adata1, adata2, adata3: AnnData objects to plot.
    - color_column (str): Column in `.obs` to use for coloring.
    - title1, title2, title3 (str): Titles for each subplot.
    - save_path (str, optional): File path to save the figure. If None, it won’t save the figure.
    
    Returns:
    - None: Displays the combined plot with a single legend.
    """
    # Define the default_26 color palette used by scanpy
    default_26 = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", 
        "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", 
        "#c5b0d5", "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5", "#393b79", 
        "#637939", "#8c6d31", "#843c39", "#7b4173", "#5254a3"
    ]
    
    # Get all unique categories across the three AnnData objects
    unique_categories = (
        set(adata1.obs[color_column].unique())
        .union(adata2.obs[color_column].unique())
        .union(adata3.obs[color_column].unique())
    )

    # Map the colors to the unique categories and sort by category name
    palette = default_26[:len(unique_categories)]  # Adjust if there are more than 26 categories
    category_colors = dict(zip(sorted(unique_categories), palette))  # Sort by category name

    # Assign the color map to each AnnData object
    adata1.uns[f"{color_column}_colors"] = [category_colors[cat] for cat in sorted(adata1.obs[color_column].cat.categories)]
    adata2.uns[f"{color_column}_colors"] = [category_colors[cat] for cat in sorted(adata2.obs[color_column].cat.categories)]
    adata3.uns[f"{color_column}_colors"] = [category_colors[cat] for cat in sorted(adata3.obs[color_column].cat.categories)]

    # Create a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot UMAP for each dataset
    sc.pl.umap(adata1, color=[color_column], title=title1, ax=axes[0], legend_loc= "none", show=False)
    sc.pl.umap(adata2, color=[color_column], title=title2, ax=axes[1], legend_loc = "none", show=False)
    sc.pl.umap(adata3, color=[color_column], title=title3, ax=axes[2], legend_loc = "none", show=False)

    # Create a single legend using alphabetically sorted categories and colors
    legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=category_colors[cat], markersize=10, label=cat) for cat in sorted(unique_categories)]
    fig.legend(handles=legend_elements, title=color_column, loc="center right", bbox_to_anchor=(1.05, 0.5))

    # Adjust layout and optionally save the figure
    plt.tight_layout(rect=[0, 0, 0.9, 1])  # Leave space on the right for the legend
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
plot_combined_umap_apoptosis(adata_apoptosis_DINO, apoptosis_adata_DP, apoptosis_adata_CP, color_column="Metadata_cmpdName", save_path="apoptosis_combined_umap.png")


## Aggregated analysis

Select if Apoptosis only aggregated or full profiles. For full profiles, can be imported like this:

In [ ]:
agg_dino = pl.read_parquet("aggregate_profiles_DINO_adjusted.parquet")
agg_dp = pl.read_parquet("aggregate_profiles_DP_aggregated.parquet")
agg_cp = pl.read_parquet("aggregate_profiles_CP_aggregated.parquet")

In [ ]:
def calculate_group_medians(adata, group_columns):
    # Ensure the columns exist in obs
    for col in group_columns:
        if col not in adata.obs:
            raise ValueError(f"Column {col} not found in obs.")
    
    # Group by the specified columns
    group_keys = adata.obs[group_columns].astype(str).agg("_".join, axis=1)
    adata.obs['group_key'] = group_keys
    
    # Calculate the median per group
    grouped = adata.to_df().groupby(adata.obs['group_key']).median()
    
    # Retrieve the unique combinations of group columns for obs
    unique_obs = adata.obs.drop_duplicates(subset='group_key')[['group_key'] + group_columns].set_index('group_key')
    unique_obs = unique_obs.loc[grouped.index]
    
    return grouped, unique_obs

group_columns = ['Metadata_Plate', 'Metadata_Site', "Metadata_Well", "moa", "Metadata_cmpdName", "Metadata_cmpdConc"]  # Replace with your column names

In [ ]:
median_data, new_obs = calculate_group_medians(adata_DINO, group_columns)
# If profiles are loaded directly like above, just replace here with creating an adata from the imported profiles
adata_DINO_agg = sc.AnnData(X=median_data.values, obs=new_obs)
adata_DINO_agg.var_names = adata_DINO_agg.var_names

In [ ]:
median_data, new_obs = calculate_group_medians(adata_DP, group_columns)
# If profiles are loaded directly like above, just replace here with creating an adata from the imported profiles
adata_DP_agg = sc.AnnData(X=median_data.values, obs=new_obs)
adata_DP_agg.var_names = adata_DP_agg.var_names

In [ ]:
median_data, new_obs = calculate_group_medians(adata_CP, group_columns)
# If profiles are loaded directly like above, just replace here with creating an adata from the imported profiles
adata_CP_agg = sc.AnnData(X=median_data.values, obs=new_obs)
adata_CP_agg.var_names = adata_CP.var_names

In [ ]:
run_scanpy(apoptosis_adata_DP_agg)
run_scanpy(apoptosis_adata_CP_agg)
run_scanpy(apoptosis_adata_DINO_agg)

In [ ]:
sc.pl.umap(apoptosis_adata_CP_agg, color = "Metadata_cmpdName")
sc.pl.umap(apoptosis_adata_DP_agg, color = "Metadata_cmpdName")
sc.pl.umap(apoptosis_adata_DINO_agg, color = "Metadata_cmpdName")

# K* calculation feature extractors

In [ ]:
adata_dino_agg = ad.AnnData(X = agg_dino[ [col for col in agg_dino.columns if col not in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas(), obs = agg_dino[ [col for col in agg_dino.columns if col in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas())

In [ ]:
adata_cp_agg = ad.AnnData(X = agg_cp[ [col for col in agg_cp.columns if col not in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas(), obs = agg_cp[ [col for col in agg_cp.columns if col in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas())
adata_dp_agg = ad.AnnData(X = agg_dp[ [col for col in agg_dp.columns if col not in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas(), obs = agg_dp[ [col for col in agg_dp.columns if col in ["Metadata_Site", "Metadata_cmpdConc", "moa", "Metadata_Plate", "Metadata_Well", "Metadat_Site", "Metdata_cmpdConc", "label", "Metadata_cmpdName"]]].to_pandas())

In [ ]:
adata_dino_samp = sc.pp.subsample(adata_DINO, n_obs = 10000, copy = True)
adata_cp_samp = sc.pp.subsample(adata_CP, n_obs = 10000, copy = True)
adata_dp_samp = sc.pp.subsample(adata_DP, n_obs = 10000, copy = True)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import skew
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def check_heterogeneity(row):
    """Find the index of the first label that's different from the first one."""
    for i, val in enumerate(row):
        if val != row[0]:
            return i
    return len(row)  # fallback if all labels are the same

def analyze_kstar(adata, feature_key="X", label_key="label", class_labels=None, skew_threshold=0.5):
    # Extract features and labels
    if feature_key == "X":
        X = adata.X
    else:
        X = adata.obsm[feature_key]
    y = adata.obs[label_key].values
    unique_labels = np.unique(y)
    
    # Pairwise distances and nearest neighbors
    dists = cdist(X, X, metric="cosine")
    sort_idx = np.argsort(dists, axis=1)
    sorted_labels = np.take(y, sort_idx)

    # Compute k* values
    kvalues = np.array([check_heterogeneity(row) for row in tqdm(sorted_labels)]).astype(np.float32)
    
    # Normalize per class
    skew_data, pattern, mean, std = [], [], [], []
    for cls in unique_labels:
        idx = np.where(y == cls)[0]
        n = len(idx)
        kvalues[idx] = kvalues[idx] / n
        c_k = kvalues[idx]
        
        mean.append(c_k.mean())
        std.append(c_k.std())
        
        s = skew(c_k)
        if np.isnan(s):
            s = -np.inf if c_k[0] == 1.0 else (np.inf if c_k[0] == 0.0 else 0.0)
        skew_data.append(s)

        if s < -skew_threshold:
            pattern.append("Clustered")
        elif s > skew_threshold:
            pattern.append("Fractured")
        else:
            pattern.append("Overlapped")

    # Safe skew values
    skew_data = np.nan_to_num(skew_data, nan=0.0, 
                              posinf=np.nanmax(skew_data), 
                              neginf=np.nanmin(skew_data))

    # Create result DataFrame
    df_kstar = pd.DataFrame({
        "k*": kvalues,
        "Label": y
    })

    summary_df = pd.DataFrame({
    "Class": unique_labels,
    "k*_mean": mean,
    "k*_std": std,
    "k*_skew": skew_data,
    "Pattern": pattern
    })
    
    return df_kstar, skew_data, pattern, summary_df


In [ ]:
df_kstar_agg, skew_values_agg, patterns_agg, summary_df_agg = analyze_kstar(
    adata=adata_dino_agg,
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
df_kstar_agg_cp, skew_values_agg_cp, patterns_agg_cp, summary_df_agg_cp = analyze_kstar(
    adata=adata_cp_agg,
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
df_kstar_agg_dp, skew_values_agg_dp, patterns_agg_dp, summary_df_agg_dp = analyze_kstar(
    adata=adata_dp_agg,
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
df_kstar, skew_values, patterns, summary_df = analyze_kstar(
    adata=sc.pp.subsample(adata_DINO, n_obs = 20000, copy = True),
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
df_kstar_dp , skew_values_dp, patterns_dp, summary_df_dp = analyze_kstar(
    adata=sc.pp.subsample(adata_DP, n_obs = 20000, copy = True),
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
df_kstar_cp, skew_values_cp, patterns_cp, summary_df_cp = analyze_kstar(
    adata=sc.pp.subsample(adata_CP, n_obs = 20000, copy = True),
    feature_key="X",  # or whatever your key is
    label_key="moa",           # or "label" depending on your column
    class_labels=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer']  # optional
)

In [ ]:
def plot_k_dist(df, skew_data, labels):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Custom color palette
    class_colors = {
        'autophagy inducer': (0.121, 0.467, 0.706),
        'apoptosis': (1.0, 0.498, 0.055),
        'ferroptosis inducer': (0.173, 0.627, 0.173),
        'immunogenic cell death': (0.839, 0.153, 0.157),
        'pyroptosis inducer': (0.580, 0.404, 0.741),
        'necroptosis inducer': (0.549, 0.337, 0.294)
    }

    def format_label(label):
        return label.replace(" inducer", "").replace("Inducer", "")

    # Prepare labels
    plot_labels = np.unique(labels)
    plot_labels_clean = [format_label(lbl) for lbl in plot_labels]

    df_plot = df[df["Label"].isin(plot_labels)].copy()
    df_plot["LabelClean"] = df_plot["Label"].apply(format_label)

    clean_palette = {format_label(k): v for k, v in class_colors.items()}

    fig, ax = plt.subplots(figsize=(10, len(plot_labels) * 0.8))

    # Draw outline violin plot with matching colors
    sns.violinplot(
        data=df_plot, y="LabelClean", x="k*", order=plot_labels_clean,
        orient='h', cut=0, inner=None, fill=False, saturation=1,
        scale='width', bw=0.2, linewidth=1, ax=ax,
        palette=clean_palette
    )

    # Draw boxenplot with the same palette and fix edge colors
    sns.boxenplot(
        data=df_plot, y="LabelClean", x="k*", order=plot_labels_clean,
        orient='h', width=0.1, linewidth=1.2,
        palette=clean_palette,
        flier_kws={'facecolor': "none", 'edgecolor': "none", 'linewidth': 0.5},
        ax=ax
    )

    for patch, label in zip(ax.patches, plot_labels_clean):
        color = clean_palette[label]
        patch.set_edgecolor(color)
        patch.set_facecolor(color)
        patch.set_linewidth(1.2)

    # Adjust axis range
    min_k, max_k = df_plot["k*"].min(), df_plot["k*"].max()
    range_k = max_k - min_k
    buffer = 0.05 * range_k if range_k > 0 else 0.01
    ax.set_xlim(min_k - buffer, max_k + buffer)
    ax.set_xlim(0, 1)

    ax.set_aspect('auto')

    ax.set(title="k* Distributions", xlabel="k*", ylabel="Class")
    ax.grid(False)
    ax.set_axisbelow(False)

    plt.tight_layout(rect=[0, 0, 0.95, 1])
    plt.show()


In [ ]:
def plot_k_dist_grid(df_list, skew_list, label_list, titles):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Custom color palette
    class_colors = {
        'autophagy inducer': (0.121, 0.467, 0.706),
        'apoptosis': (1.0, 0.498, 0.055),
        'ferroptosis inducer': (0.173, 0.627, 0.173),
        'immunogenic cell death': (0.839, 0.153, 0.157),
        'pyroptosis inducer': (0.580, 0.404, 0.741),
        'necroptosis inducer': (0.549, 0.337, 0.294)
    }

    def format_label(label):
        return label.replace(" inducer", "").replace("Inducer", "")

    # Clean class names and palettes
    all_labels = label_list
    clean_palette = {format_label(k): v for k, v in class_colors.items()}
    plot_labels_clean = [format_label(lbl) for lbl in all_labels]

    n_panels = len(df_list)
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, len(all_labels) * 0.8), sharey=True)

    for i in range(n_panels):
        df = df_list[i].copy()
        df = df[df["Label"].isin(all_labels)]
        df["LabelClean"] = df["Label"].apply(format_label)

        ax = axes[i]

        sns.violinplot(
            data=df, y="LabelClean", x="k*", order=plot_labels_clean,
            orient='h', cut=0, inner=None, fill=False, saturation=1,
            scale='width', bw=0.2, linewidth=1, ax=ax,
            palette=clean_palette
        )

        sns.boxenplot(
            data=df, y="LabelClean", x="k*", order=plot_labels_clean,
            orient='h', width=0.1, linewidth=1.2,
            palette=clean_palette,
            flier_kws={'facecolor': "none", 'edgecolor': "none", 'linewidth': 0.5},
            ax=ax
        )

        for patch, label in zip(ax.patches, plot_labels_clean):
            color = clean_palette[label]
            patch.set_edgecolor(color)
            patch.set_facecolor(color)
            patch.set_linewidth(1.2)

        # X-axis limits based on individual data
        min_k, max_k = df["k*"].min(), df["k*"].max()
        range_k = max_k - min_k
        buffer = 0.05 * range_k if range_k > 0 else 0.01
        ax.set_xlim(min_k - buffer, max_k + buffer)
        ax.set_xlim(min_k - buffer, 1)
        ax.set_aspect('auto')

        ax.set(title=titles[i], xlabel="k*")
        if i == 0:
            ax.set_ylabel("Class")
        else:
            ax.set_ylabel("")
        ax.grid(False)
        ax.set_axisbelow(False)

    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()


In [ ]:
plot_k_dist_grid(
    df_list=[df_kstar_cp, df_kstar_dp, df_kstar],
    skew_list=[skew_values_cp, skew_values_dp, skew_values],
    label_list=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer'],
    titles=["CellProfiler", "DeepProfiler", "DINO"]
)

In [ ]:
plot_k_dist_grid(
    df_list=[df_kstar_agg_cp, df_kstar_agg_dp, df_kstar_agg],
    skew_list=[skew_values_agg_cp, skew_values_agg_dp, skew_values_agg],
    label_list=['immunogenic cell death', 'pyroptosis inducer', 'ferroptosis inducer', 'apoptosis', 'necroptosis inducer', 'autophagy inducer'],
    titles=["CellProfiler", "DeepProfiler", "DINO"]
)

# Dose-response analysis

In [ ]:
comp_list_paper = ["CLADRIBINE", "SN 38", "TOPOTECAN"]

In [ ]:
import scanpy as sc
import numpy as np

def run_density_estimation(data, query_compound, umap_key='X_umap', max_cells=5000, target_cells=5000, min_cells=100):
    """
    Runs density estimation on an AnnData object, highlighting the density of a specific compound concentration.
    
    Parameters:
    data (AnnData): AnnData object containing the UMAP and metadata.
    query_compound (str): The compound name to highlight.
    umap_key (str): Key in `.obsm` for the UMAP coordinates (default is 'X_umap').
    max_cells (int): Maximum number of cells allowed in a group before sampling (default is 5000).
    target_cells (int): Target number of cells to sample down to if a group exceeds max_cells (default is 5000).
    min_cells (int): Minimum number of cells required to include a group (default is 100).

    Returns:
    None, directly plots the density estimation map.
    """
    # Step 1: Create a unique identifier for the compound + concentration
    data = data.copy()
    data.obs["Metadata_cmpdNameConc2"] = data.obs["Metadata_cmpdName"].astype(str) + "_" + data.obs["Metadata_cmpdConc"].astype(str)
    data.obs['Metadata_cmpdConc'] = data.obs['Metadata_cmpdConc'].astype("str").astype('category')
    data = data[data.obs["Metadata_cmpdConc"].isin(["0.1", "0.3", "1.0", "5.0"])].copy()
    # Step 2: Create a new column where only the selected compound concentration is kept
    data.obs["Density_Label"] = data.obs.apply(
        lambda row: row["Metadata_cmpdNameConc2"] if row["Metadata_cmpdName"] == query_compound else "Apoptosis embedding", axis=1
    )
    data.obs['Density_Label'] = data.obs['Density_Label'].astype('category')

    # Step 3: Sample based on the `Density_Label` column
    sampled_indices = []
    for label in data.obs["Density_Label"].unique():
        label_cells = data.obs["Density_Label"] == label
        num_cells = label_cells.sum()
        if num_cells < min_cells:
            print(f"Skipping group '{label}' as it has only {num_cells} cells, which is below the minimum threshold of {min_cells}.")
            continue

        if num_cells > max_cells:  # Only sample for the compound of interest
            # Sample down to target number of cells for the query compound
            sampled_indices.extend(np.random.choice(data[label_cells].obs_names, size=target_cells, replace=False))
        else:
            # Keep all cells if below the threshold or it's the "Other compound" group
            sampled_indices.extend(data[label_cells].obs_names)

    # Create a new AnnData object with the sampled cells
    data = data[sampled_indices].copy()

    # Step 4: Run scanpy's density estimation on the new grouping
    sc.tl.embedding_density(data, basis='umap', groupby='Density_Label')
    sc.pl.embedding_density(
        data, 
        bg_dotsize=40, 
        fg_dotsize=150, 
        basis='umap', 
        key='umap_density_Density_Label', 
        groups=[label for label in data.obs["Density_Label"].cat.categories if label != "Apoptosis embedding"],
        ncols=7
    )

### CellProfiler

In [ ]:
for comp in comp_list_paper:
    print(comp)
    run_density_estimation(apoptosis_adata_CP, comp)

### DeepProfiler

In [ ]:
for comp in comp_list_paper:
    print(comp)
    run_density_estimation(apoptosis_adata_DP, comp)

### DINO

In [ ]:
for comp in comp_list:
    print(comp)
    run_density_estimation(adata_apoptosis, comp)

## Calculate etest/edist between compound concentrations (Panel D Fig. 4)

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from statsmodels.stats.multitest import multipletests


def pairwise_etest(adata, groupby, sample_size=100, n_permutations=1000):
    """
    Run pairwise E-tests between all unique groups in adata.obs[groupby],
    using scperturb.etest, and return a summary DataFrame.

    Parameters:
        adata: AnnData object with PCA embeddings in .obsm['X_pca']
        groupby: obs column with group labels (e.g., 'Metadata_cmpdNameConc')
        sample_size: number of cells to subsample per group
        n_permutations: number of permutations

    Returns:
        DataFrame with: group1, group2, edist, pvalue, pvalue_adj, significant_adj
    """
    adata.obs_names = adata.obs_names.astype(str)
    results = []

    groups = adata.obs[groupby].unique()
    for g1, g2 in combinations(groups, 2):
        subset = adata[adata.obs[groupby].isin([g1, g2])].copy()

        # Make sure both groups have enough samples
        min_size = min((subset.obs[groupby] == g1).sum(), (subset.obs[groupby] == g2).sum())
        if min_size < sample_size:
            continue

        try:
            df = etest(
                subset,
                obs_key=groupby,
                obsm_key='X_pca',
                dist='sqeuclidean',
                control=g1,
                alpha=0.05,
                runs=n_permutations,
                n_jobs=-1
            )

            # Grab the row for the comparison group (the one not used as control)
            row = df.loc[g2] if g2 in df.index else df.loc[g1]

            results.append({
                "group1": g1,
                "group2": g2,
                "edist": row["edist"],
                "pvalue": row["pvalue"],
                "pvalue_adj": row["pvalue_adj"],
                "significant_adj": row["significant_adj"]
            })

        except Exception as e:
            print(f"Failed on {g1} vs {g2}: {e}")

    return pd.DataFrame(results)

In [ ]:
from scperturb import *
def run_etest(adata, compound_col, contr, sample_size, permut_num):  
    print("Data imported")
    adata.obs_names = adata.obs_names.astype(str)
    anndata_group = equal_subsampling(adata, "Metadata_cmpdNameConc", N_min= sample_size)
    print("Running etest!")
    df = etest(anndata_group, obs_key=compound_col, obsm_key='X_pca', dist='sqeuclidean', control=contr, alpha=0.05, runs=permut_num, n_jobs=-1)
    print(df)
    print("Saving test plots.")
    
    df.loc[df.index==contr, 'significant_adj'] = contr
    df['neglog10_pvalue_adj'] = -np.log10(df['pvalue_adj'])
    with sns.axes_style('whitegrid'):
        sns.scatterplot(data=df, y='neglog10_pvalue_adj', x='edist', hue='significant_adj', palette={True: 'tab:green', False: 'tab:red', contr: 'tab:orange'}, s=30)
    plt.title('E-test results')
    plt.xlabel('E-distance from control')
    plt.ylabel('E-test neg log10 of adjusted p-value')
    plt.savefig(f"etest_res_{sample_size}_samples_{permut_num}_perms_grit_all.png")
    plt.show()
    df2 = pd.DataFrame(df)
    #bool_map = {
    #'[DMSO]': False,
    # Add other mappings as necessary
    #}
    df2['significant_adj'] = df2['significant_adj'].map(bool_map).astype(bool)
 
    return df2

In [ ]:
apoptosis_adata_DP.obs["Metadata_cmpdNameConc"] = apoptosis_adata_DP.obs["Metadata_cmpdName"].astype("str") + "_" + apoptosis_adata_DP.obs["Metadata_cmpdConc"].astype("str") 
apoptosis_adata_CP.obs["Metadata_cmpdNameConc"] = apoptosis_adata_CP.obs["Metadata_cmpdName"].astype("str") + "_" + apoptosis_adata_CP.obs["Metadata_cmpdConc"].astype("str") 
adata_apoptosis_DINO.obs["Metadata_cmpdNameConc"] = adata_apoptosis_DINO.obs["Metadata_cmpdName"].astype("str") + "_" + adata_apoptosis_DINO.obs["Metadata_cmpdConc"].astype("str") 

In [ ]:
edist_dict_DP = {}
etest_dict_DP = {}
comps = ["CLADRIBINE", "SN 38", "TOPOTECAN"]
for c in comps:
    adata_filt = apoptosis_adata_DP[
        (apoptosis_adata_DP.obs["Metadata_cmpdName"] == c) &
        (apoptosis_adata_DP.obs["Metadata_cmpdConc"].isin([0.1, 0.3, 1.0, 5.0]))
    ].copy()
    estats = edist(adata_filt, "Metadata_cmpdNameConc", obsm_key='X_pca', dist='sqeuclidean', n_jobs= -1)
    edist_dict_DP[c] = estats
    res_df = pairwise_etest(adata_filt, groupby="Metadata_cmpdNameConc", sample_size=100, n_permutations=1000)
    etest_dict_DP[c] = res_df

In [ ]:
edist_dict_CP = {}
etest_dict_CP = {}
comps = ["CLADRIBINE", "SN 38", "TOPOTECAN"]
for c in comps:
    valid_concs = [0.1, 0.3, 1.0, 5.0]
    mask = apoptosis_adata_CP.obs["Metadata_cmpdName"] == c

    conc_mask = np.isin(
        np.round(apoptosis_adata_CP.obs["Metadata_cmpdConc"].astype(float), 2), 
        valid_concs
    )

    adata_filt = apoptosis_adata_CP[mask & conc_mask].copy()

    estats = edist(adata_filt, "Metadata_cmpdNameConc", obsm_key='X_pca', dist='sqeuclidean', n_jobs= -1)
    edist_dict_CP[c] = estats
    res_df = pairwise_etest(adata_filt, groupby="Metadata_cmpdNameConc", sample_size=100, n_permutations=1000)
    etest_dict_CP[c] = res_df

100%|██████████| 1000/1000 [00:29<00:00, 33.81it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:25<00:00, 39.22it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:14<00:00, 67.71it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:25<00:00, 38.50it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:16<00:00, 61.27it/s]


In [ ]:
edist_dict_DINO = {}
etest_dict_DINO = {}
comps = ["CLADRIBINE", "SN 38", "TOPOTECAN"]
for c in comps:
    print(c)
    valid_concs = [0.1, 0.3, 1.0, 5.0]
    mask = adata_apoptosis_DINO.obs["Metadata_cmpdName"] == c

    conc_mask = np.isin(
        np.round(adata_apoptosis_DINO.obs["Metadata_cmpdConc"].astype(float), 2), 
        valid_concs
    )

    adata_filt = adata_apoptosis_DINO[mask & conc_mask].copy()

    estats = edist(adata_filt, "Metadata_cmpdNameConc", obsm_key='X_pca', dist='sqeuclidean', n_jobs= -1)
    etest_dict_DINO[c] = estats
    res_df = pairwise_etest(adata_filt, groupby="Metadata_cmpdNameConc", sample_size=100, n_permutations=1000)
    etest_dict_DINO[c] = res_df

100%|██████████| 1000/1000 [00:06<00:00, 161.64it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:04<00:00, 208.71it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:06<00:00, 152.13it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:01<00:00, 672.69it/s]
/home/jovyan/share/data/analyses/benjamin/.venv/lib/python3.10/site-packages/statsmodels/stats/multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
100%|██████████| 1000/1000 [00:03<00:00, 305.16i

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import re

def extract_compound_and_conc(label):
    m = re.match(r"(.+?)_([\d.]+)$", label)
    return (m.group(1), float(m.group(2))) if m else (label, np.nan)

def sort_by_compound_and_conc(labels):
    parsed = [(*extract_compound_and_conc(lbl), lbl) for lbl in labels]
    return [x[2] for x in sorted(parsed, key=lambda x: (x[0], x[1]))]

def plot_multi_heatmap_matrix(edist_dict, ptest_dict):
    """
    edist_dict : {compound: square DataFrame of E-distances, idx & cols = 'CMPD_CONC'}
    ptest_dict: {compound: long DataFrame with columns ['group1','group2','pvalue_adj']}
    """
    compounds = list(edist_dict.keys())
    n = len(compounds)
    fig, axes = plt.subplots(1, n, figsize=(6.5*n, 6), dpi=300)
    if n == 1: axes = [axes]

    # determine shared color scale
    all_d = np.concatenate([df.values.flatten() for df in edist_dict.values()])
    vmin, vmax = np.nanmin(all_d), np.nanmax(all_d)
    log_vmin, log_vmax = np.log10(vmin+1), np.log10(vmax+1)
    cmap = sns.color_palette("rocket", as_cmap = True)

    for ax, cmpd in zip(axes, compounds):
        # 1) pull & sort your square E-dist matrix
        edf = edist_dict[cmpd]
        labels = sort_by_compound_and_conc(edf.index)
        mat = edf.loc[labels, labels]
        mat_log = np.log10(mat + 1)

        # 2) pivot your ptest long table into a square matrix
        pdf = ptest_dict[cmpd][['group1','group2','pvalue_adj']].copy()
        # ensure both orders exist so the matrix is symmetric
        pdf_swap = pdf.rename(columns={'group1':'group2','group2':'group1'})
        pdf = pd.concat([pdf, pdf_swap], ignore_index=True)
        pmat = pdf.pivot(index='group1', columns='group2', values='pvalue_adj')
        pmat = pmat.reindex(index=labels, columns=labels).fillna(1.0)

        # 3) draw the heatmap
        sns.heatmap(
            mat_log,
            ax=ax,
            cmap=cmap,
            vmin=log_vmin, vmax=log_vmax,
            cbar=False,
            linewidths=0.5, linecolor='lightgray',
            xticklabels=True, yticklabels=True
        )

        # 4) overlay asterisks
        size = len(labels)
        for i in range(size):
            for j in range(size):
                p = pmat.iat[i,j]
                if p < 0.001: stars = '***'
                elif p < 0.01: stars = '**'
                elif p < 0.05: stars = '*'
                else: stars = ''
                if stars:
                    ax.text(
                        j+0.5, i+0.5, stars,
                        ha='center', va='center',
                        color='black', fontsize=16, fontweight='bold'
                    )

        # 5) clean up axes
        doses = [lbl.split('_')[-1] for lbl in labels]
        ax.set_xticklabels(doses, rotation=0, ha='center', fontsize=9)
        ax.set_yticklabels(doses, rotation=0, fontsize=9)
        ax.set_xlabel('Dose', fontsize=11)
        ax.set_ylabel('Dose' if ax is axes[0] else '', fontsize=11)
        ax.set_title(cmpd, fontsize=14, fontweight='bold')

    # shared colorbar
    cax = fig.add_axes([0.92, 0.30, 0.02, 0.4])
    norm = plt.Normalize(log_vmin, log_vmax)
    sm   = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("log₁₀(E-dist + 1)", fontsize=12)

    plt.subplots_adjust(wspace=0.3, right=0.88, bottom=0.15, top=0.9)
    plt.show()


In [ ]:
plot_multi_heatmap_matrix(edist_dict_DP, etest_dict_DP)


In [ ]:
plot_multi_heatmap_matrix(edist_dict_CP, etest_dict_CP)


In [ ]:
plot_multi_heatmap_matrix(edist_dict_DINO, etest_dict_DINO)


# Correlation matrix (Panel C, Fig. 1)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme()

def create_pca_df(adata, n_pcs=50):
    """
    Extract PCA coordinates and metadata from the AnnData object.
    """
    # Extract the first n_pcs PCA coordinates
    pca_coords = pd.DataFrame(adata.obsm['X_pca'][:, :n_pcs], index=adata.obs.index)
    
    # Extract relevant metadata columns ('id' and 'moa')
    pca_coords['id'] = adata.obs['id'].values
    pca_coords['moa'] = adata.obs['moa'].values
    
    return pca_coords

def correlation_heatmap(adata, save_dir):
    pca_df = pd.DataFrame(
        adata_agg.obsm['X_pca'][:, :50],  # Extract the first n_pcs components
        index=adata_agg.obs_names
    )
        
        # Add metadata columns (assuming 'id' and 'moa' are stored in adata.obs)
    pca_df['id'] = adata_agg.obs['id'].values
    pca_df['moa'] = adata_agg.obs['moa'].values.astype(str)

        # Step 2: Calculate correlation matrix between rows (IDs)
    corr_matrix = pca_df.iloc[:, :50].T.corr()

        # Step 3: Prepare color annotations for the 'moa' column
    unique_moas = pca_df['moa'].unique()
    palette = sns.color_palette("husl", len(unique_moas))
    #moa_color_map = dict(zip(unique_moas, palette))
    moa_color_map = {'autophagy inducer': (0.12156862745098039, 0.4666666666666667, 0.7058823529411765), 'apoptosis': (1.0, 0.4980392156862745, 0.054901960784313725), 'ferroptosis inducer': (0.17254901960784313, 0.6274509803921569, 0.17254901960784313), 'immunogenic cell death': (0.8392156862745098, 0.15294117647058825, 0.1568627450980392), 'pyroptosis inducer': (0.5803921568627451, 0.403921568627451, 0.7411764705882353), 'necroptosis inducer': (0.5490196078431373, 0.33725490196078434, 0.29411764705882354)}


        # Map colors to the 'moa' annotations
    moa_colors = pca_df['moa'].map(moa_color_map)

    g = sns.clustermap(
        corr_matrix,
        center=0,
        cmap='vlag',
        row_colors=moa_colors,
        col_colors=moa_colors,
        dendrogram_ratio=(.1, .2),
        cbar_pos=(0.02, 0.8, 0.05, 0.18),
        figsize=(12, 13)
    )
    g.ax_row_dendrogram.remove()
    g.ax_heatmap.set_xticklabels([])
    g.ax_heatmap.set_yticklabels([])

    # Optionally, remove axis labels entirely
    g.ax_heatmap.set_xlabel("")
    g.ax_heatmap.set_ylabel("")
    plt.savefig(save_dir, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
correlation_heatmap(adata_agg_DINO) #use aggregated adata from above

In [ ]:
correlation_heatmap(adata_agg_cp) #use aggregated adata from above

In [ ]:
correlation_heatmap(adata_agg_dp) #use aggregated adata from above

# Grit calculation (used for filtering, script with code on github)

## Exemplary calculation here, can be re-created for each feature set. Results are on figshare)

In [ ]:
import polars as pl
from grit_script import *
import tqdm

In [ ]:
df_X = pd.DataFrame(adata_DP.X.toarray(), index=adata_DP.obs_names, columns=adata_DP.var_names)
df_obs = adata_DP.obs.copy()
df_combined = pd.concat([df_obs, df_X], axis=1).reset_index(drop=True)


In [ ]:
#df_combined = df_combined.iloc[:, 1:].reset_index(drop = True)
df_combined = df_combined.apply(lambda col: col.astype(str) if col.dtype.name == 'category' else col)


In [ ]:
df_combined.columns = [col if i < 13 else f'Feature_{i-13}' for i, col in enumerate(df_combined.columns)]
feature_df = pl.DataFrame(df_combined)

In [ ]:
features_fixed = [feat for feat in feature_df.columns if "Feature" in feat]
feature_df_agg = (
feature_df.group_by(["moa", "Metadata_Plate", 'Metadata_Well', 'Metadata_cmpdName', "Metadata_cmpdConc", "Metadata_Site"])
    .agg([pl.col(feature).median().alias(feature) for feature in features_fixed])
)
feature_df_agg = feature_df_agg.with_columns((pl.col("Metadata_cmpdName") + "_" + pl.col("Metadata_cmpdConc").cast(pl.Utf8) + "_" + pl.col("Metadata_Well") + "_" + pl.col("Metadata_Site")).alias("compound_replicate"))

In [ ]:
feature_df_agg = feature_df_agg.with_columns(
    pl.concat_str(
        [pl.col("Metadata_cmpdName"), pl.col("Metadata_cmpdConc")],
        separator="_"
    ).alias("Metadata_cmpdConcName")
)

In [ ]:
def compute_grit(dfZscores):
    plates = list(dfZscores["Metadata_Plate"].unique())
    features_fixed = [feat for feat in dfZscores.columns if "Feature" in feat]
    meta_features = [feat for feat in dfZscores.columns if feat not in features_fixed]
    grit_score_list = []
    for p in tqdm.tqdm(plates): 
        profiles = dfZscores[dfZscores["Metadata_Plate"] == p]
        grit_scores = evaluate(
                profiles=profiles,  
                features=features_fixed,
                meta_features=meta_features,   
                replicate_groups={"profile_col": "compound_replicate", "replicate_group_col": "Metadata_cmpdConcName"},
                operation="grit",
                similarity_metric="pearson",
                grit_replicate_summary_method="median", # median
                grit_control_perts=profiles.query("Metadata_cmpdConcName == 'DIMETHYL SULFOXIDE_0.1'").compound_replicate.unique().tolist()
            ).assign(p = p)
        grit_score_list.append(grit_scores)
    return grit_score_list

In [ ]:
grits_aggregated = compute_grit(feature_df_agg.to_pandas())


In [ ]:
grit_scores_total = pd.concat(grits_aggregated)
grit_compound_final = grit_scores_total.groupby("group").mean("grit").reset_index()
grit_filtered_1 = grit_scores_total[grit_scores_total["grit"] > 1.5]
#grit_filtered.to_csv("grit_over1.5_specs5k_moa.csv")
grit_filtered_1["filter"] = grit_filtered_1["perturbation"] + "_" + grit_filtered_1["p"]
grit_comps = list(grit_filtered_1["filter"])

# Training data split

## Same for all feature sets, and use full_split df to split aggregated 

In [ ]:
import random
import polars as pl

random.seed(69)

def well_split(df, moa_column='moa', train_frac=0.6, test_frac=0.2, val_frac=0.2):
    # Remove unused categories from the 'moa' column
    df = df.with_columns(pl.col(moa_column).cast(pl.Categorical).cast(pl.Categorical))

    # Get unique MOA categories
    moa_broad = df.select(moa_column).unique().to_series().to_list()

    train_list = []
    valid_list = []
    test_list = []

    for m in moa_broad:
        moa_df = df.filter(pl.col(moa_column) == m)

        # Group by compound and wells
        moa_grouped = moa_df.group_by(["moa", "Metadata_cmpdName", "Metadata_Well", "Metadata_Plate"]).agg(
            pl.count().alias('Compound Count')
        )

        # Calculate compound fraction
        total_count = moa_grouped['Compound Count'].sum()
        moa_grouped = moa_grouped.with_columns(
            (pl.col('Compound Count') / total_count).alias('Compound fraction')
        )

        # Shuffle rows
        moa_grouped = moa_grouped.sample(fraction=1, seed=42)

        current_train_sum = 0
        current_valid_sum = 0

        for row in moa_grouped.iter_rows():
            if current_train_sum < train_frac:
                train_list.append(row)
                current_train_sum += row[-1]  # Compound fraction
            elif current_valid_sum < val_frac:
                valid_list.append(row)
                current_valid_sum += row[-1]  # Compound fraction
            else:
                test_list.append(row)

    # Convert lists to DataFrames
    train_df = pl.DataFrame(train_list, schema=moa_grouped.schema)
    valid_df = pl.DataFrame(valid_list, schema=moa_grouped.schema)
    test_df = pl.DataFrame(test_list, schema=moa_grouped.schema)

    # Add split labels
    train_df = train_df.with_columns(pl.lit("training").alias("split"))
    valid_df = valid_df.with_columns(pl.lit("valid").alias("split"))
    test_df = test_df.with_columns(pl.lit("test").alias("split"))

    # Combine all into one final DataFrame
    full_split = pl.concat([train_df, valid_df, test_df])

    return full_split, moa_grouped, moa_df



In [ ]:
full_split_CP, moa_grouped_cp, moa_df_cp = well_split(cp_sc_profiles)
#full_split.write_csv("data_revision/split_CellProfiler_moa.csv")

In [ ]:
full_split_DP, moa_grouped_DP, moa_df_DP = well_split(dp_sc_profiles)
#full_split_DP.write_csv("data_revision/split_DeepProfiler_moa.csv")

In [ ]:
full_split_DINO, moa_grouped, moa_df = well_split(DINO_sc_profiles)
#full_split_DINO.write_csv("data_revision/split_DINO_moa.csv")

In [ ]:
def split_dataframe(input_df, split_df):
    # Ensure the split_df DataFrame has the necessary columns
    required_columns = {"Metadata_cmpdName", "Metadata_Well", "Metadata_Plate", "split"}
    if not required_columns.issubset(split_df.columns):
        raise ValueError(f"Split DataFrame must have columns: {required_columns}")
    input_df = input_df.with_columns([
    pl.col(pl.Categorical).cast(pl.Utf8)  # Convert all columns of type 'Categorical' to 'Utf8' (string)
    ])
    split_df = split_df.with_columns([
    pl.col(pl.Categorical).cast(pl.Utf8)  # Convert all columns of type 'Categorical' to 'Utf8' (string)
    ])
    # Join the input DataFrame with the split DataFrame on 'Metadata_cmpdName', 'Metadata_Well', and 'Metadata_Plate'
    merged_df = input_df.join(
        split_df, on=["Metadata_cmpdName", "Metadata_Well", "Metadata_Plate"], how="inner"
    )

    # Perform the split based on the 'split' column in the merged DataFrame
    train_df = merged_df.filter(pl.col("split") == "training")
    valid_df = merged_df.filter(pl.col("split") == "valid")
    test_df = merged_df.filter(pl.col("split") == "test")
    
    return train_df, valid_df, test_df

In [ ]:
train_df_CP, valid_df_CP, test_df_CP = split_dataframe(cp_sc_profiles, full_split_CÜ)

In [ ]:
train_df_DP, valid_df_DP, test_df_DP = split_dataframe(dp_sc_profiles, full_split_DP)
train_df_DINO, valid_df_DINO, test_df_DINO = split_dataframe(DINO_sc_profiles, full_split_DINO)

In [ ]:
columns_to_drop = ["Metadata_Plate", "Metadata_cmpdConc", "label", "Metadata_cmpdName", "Metadata_Site", "Metadata_Well", "moa_right", "Compound Count", "Compound fraction", "split"]

In [ ]:
import os
output_dir = "data_revision/classification_splits"

# Dictionary of datasets to save
datasets = {
    "CP": [train_df_CP, valid_df_CP, test_df_CP],
    "DP": [train_df_DP, valid_df_DP, test_df_DP],
    "DINO": [train_df_DINO, valid_df_DINO, test_df_DINO]
}

splits = ["train", "valid", "test"]

# Save files
for label, dfs in datasets.items():
    for split_name, df in zip(splits, dfs):
        filename = f"celldeath_{split_name}_{label}_singlecell.parquet"
        path = os.path.join(output_dir, filename)
        df_filtered = df.drop([col for col in columns_to_drop if col in df.columns])
        df_filtered.write_parquet(path)

print("✅ All files saved to:", output_dir)


In [ ]:
test_df_DINO.select(pl.exclude(columns_to_drop)).write_parquet("classification_splits/celldeath_valid_DINO_aggregated.parquet")

In [ ]:
valid_df_DINO.select(pl.exclude(columns_to_drop)).write_parquet("classification_splits/celldeath_test_DINO_aggregated.parquet")

In [ ]:
train_df_DINO.select(pl.exclude(columns_to_drop)).write_parquet("/classification_splits/celldeath_train_DINO_aggregated.parquet")

# Target analysis

In [ ]:
import pandas as pd

# Create the initial DataFrame
data = {
    "Compound": [
        "2;3-DCPE Hydrochloride", "Anatoxin A Fumarate", "Apoptosis Activator 2", 
        "AT 101", "Betulinic Acid", "BZ 423", "C75", "Camptothecin", "Cladribine", 
        "Doxorubicin", "Epirubicin", "Etoposide", "FK 866", "Fludarabine", "G5", 
        "Kaempferol", "Mitomycin C", "MPC 6827", "Narciclasine", "Oncrasin 1", 
        "Rifaximin", "SMBA 1", "SN 38", "Topotecan"
    ],
    "Mechanism of Action": [
        "Downregulates Bcl-XL protein expression.", "Agonist of nicotinic acetylcholine receptors (nAChRs).", 
        "Induces apoptosis selectively in tumor cells.", "Downregulates Bcl-2 and Mcl-1 proteins.", 
        "Directly affects mitochondria to release cytochrome c.", "Inhibits ATP synthase to induce apoptosis.", 
        "Inhibits fatty acid synthase (FASN).", "Inhibits DNA topoisomerase I.", 
        "Incorporates into DNA, inhibiting synthesis.", "Intercalates into DNA and inhibits topoisomerase II.", 
        "Similar to doxorubicin; inhibits topoisomerase II.", "Inhibits topoisomerase II, causing DNA strand breaks.", 
        "Inhibits NAMPT, depleting NAD+ levels.", "Inhibits DNA polymerase and ribonucleotide reductase.", 
        "Ubiquitin isopeptidase inhibitor; induces apoptosis", 
        "Modulates Bcl-2 family proteins to induce apoptosis.", "Alkylates DNA, causing cross-linking.", 
        "Antineoplastic; a small-molecule inhibitor of microtubule formation that is not a substrate for multidrug resistance pumps.", 
        "Inhibits eEF1A to block protein synthesis.", "Induces apoptosis in mutated KRAS cancer cells.", 
        "Binds bacterial RNA polymerase, inhibiting RNA synthesis.", 
        "High affinity and selective activator of Bax", "Inhibits topoisomerase I, causing DNA damage.", 
        "Inhibits topoisomerase I, causing DNA strand breaks."
    ],
    "Target": [
        "Bcl-XL", "nAChRs", "APAF-1", "Bcl-2", "Mitochondria", "ATP Synthase", "FASN", 
        "Topoisomerase I", "DNA Polymerase", "Topoisomerase II", "Topoisomerase II", 
        "Topoisomerase II", "NAMPT", "DNA Polymerase", "Ubiquitin isopeptidase", 
        "Bcl-2", "DNA", "TUBB", "eEF1A", "RNA Polymerase II", "RNA Polymerase", 
        "BAX (Bcl-2)", "Topoisomerase I", "Topoisomerase I"
    ]
}

# Convert to DataFrame
compound_df = pd.DataFrame(data)

# Convert "Compound" column to uppercase
compound_df["Metadata_cmpdName"] = compound_df["Compound"].str.upper()

# Rename "Mechanism of Action" column to "Description"
compound_df.rename(columns={"Mechanism of Action": "Description"}, inplace=True)

# Assume `adata` is your existing AnnData object
# Merge compound DataFrame into the obs attribute of adata
# Ensure obs has a corresponding "Compound" column for the merge
def merge_with_obs(adata, compound_df):
    if "Metadata_cmpdName" not in adata.obs:
        raise ValueError("The 'Compound' column must exist in `adata.obs` to perform the merge.")
    # Convert "Compound" in obs to uppercase for merging
    adata.obs["Metadata_cmpdName"] = adata.obs["Metadata_cmpdName"].str.upper()
    # Merge the data
    adata.obs = adata.obs.merge(compound_df, on="Metadata_cmpdName", how="left")
    return adata

# Apply the merge function (uncomment to run when `adata` exists)
# adata = merge_with_obs(adata, compound_df)

# Display the modified DataFrame
compound_df


In [ ]:
adata_apoptosis_DINO = merge_with_obs(adata_apoptosis_DINO, compound_df) # Use DINO Apoptosis df from above
adata_apoptosis_DINO = adata_apoptosis_DINO[~adata_apoptosis_DINO.obs["Target"].isin(["Mitochondria"])].copy() # exclude Mito comp due to mismatch in comps

In [ ]:
apoptosis_adata_DP = merge_with_obs(apoptosis_adata_DP, compound_df) #See above

In [ ]:
apoptosis_adata_CP = merge_with_obs(apoptosis_adata_CP, compound_df) #See above

In [ ]:
def plot_target(adata_apoptosis):
    adata_apoptosis.obs["Target"] = adata_apoptosis.obs["Target"].astype("category")
    adata_apoptosis.obs["Metadata_cmpdName"] = adata_apoptosis.obs["Metadata_cmpdName"].astype("category")

    # Count number of unique treatments per Target
    group_counts = (
        adata_apoptosis.obs
        .groupby("Target")["Metadata_cmpdName"]
        .nunique()
    )

    new_categories = {
        cat: f"{cat} (n = {group_counts[cat]})" if cat in group_counts else cat
        for cat in adata_apoptosis.obs["Target"].cat.categories
    }

    # Apply the renamed categories
    adata_apoptosis.obs["Target"] = adata_apoptosis.obs["Target"].cat.rename_categories(new_categories)

    # Plot
    sc.pl.umap(adata_apoptosis, color="Target", legend_loc="right margin")


In [ ]:
plot_target(adata_apoptosis_DINO)

In [ ]:
sc.pl.umap(adata_apoptosis, color ="Target")
sc.pl.umap(apoptosis_adata_DP, color ="Target")
sc.pl.umap(apoptosis_adata_CP, color ="Target")


In [ ]:
sc.tl.embedding_density(adata_apoptosis, basis='umap', groupby='Target')
sc.pl.embedding_density(adata_apoptosis, bg_dotsize = 40, fg_dotsize= 150, basis='umap', ncols = 5, key='umap_density_Target')

In [ ]:
sc.tl.embedding_density(apoptosis_adata_DP, basis='umap', groupby='Target')
sc.pl.embedding_density(apoptosis_adata_DP, bg_dotsize = 40, fg_dotsize= 150, basis='umap', ncols = 5, key='umap_density_Target')

In [ ]:
sc.tl.embedding_density(apoptosis_adata_CP, basis='umap', groupby='Target')
sc.pl.embedding_density(apoptosis_adata_CP, bg_dotsize = 40, fg_dotsize= 150, basis='umap', ncols = 5, key='umap_density_Target')

## Target analysis all comps

In [ ]:
import pandas as pd

table_data = {
    "Compound": [
        "Etoposide", "Betulinic Acid", "Apoptosis Activator 2", "Anatoxin A Fumarate", 
        "2;3-DCPE Hydrochloride", "AT 101", "AZD 2461", "Bleomycin", "Bortezomib",
        "BZ 423", "Camptothecin", "Carboplatin", "C 75", "Cisplatin", "Cladribine",
        "Crizotinib", "DACTINOMYCIN (ACTINOMYCIN D)", "Daunorubicin", "Doxorubicin", "Domperidone",
        "Epirubicin", "Erastin", "Everolimus", "FIN 56", "FK 866", "Fludarabine",
        "G5", "HS-173", "Kaempferol", "L-690,330", "Lovastatin", "Loperamide",
        "Mitomycin C", "MPC 6827", "Narciclasine", "Nigericin", "Oncrasin 1",
        "Oxaliplatin", "Pimozide", "Polyphyllin VI", "Pravastatin", "Rifaximin",
        "Shikonin", "Simvastatin", "SMBA 1", "SN 38", "Sorafenib", "Sulfasalazine",
        "Teniposide", "Topotecan", "VU0359595"
    ],
    "Description": [
        "Inhibits topoisomerase II, causing DNA strand breaks.", 
        "Directly affects mitochondria to release cytochrome c.",
        "Induces apoptosis selectively in tumor cells.",
        "Agonist of nicotinic acetylcholine receptors (nAChRs).",
        "Downregulates Bcl-XL protein expression.",
        "Downregulates Bcl-2 and Mcl-1 proteins.",
        "Inhibits PARP, leading to DNA damage accumulation.",
        "Induces DNA strand breaks, leading to apoptosis.",
        "Inhibits the 26S proteasome, leading to apoptosis in cancer cells.",
        "Inhibits ATP synthase to induce apoptosis.",
        "Inhibits topoisomerase I, causing DNA damage.",
        "Forms DNA cross-links, leading to apoptosis.",
        "Inhibits fatty acid synthase (FASN).",
        "Forms DNA cross-links, leading to apoptosis.",
        "Incorporates into DNA, inhibiting synthesis.",
        "Inhibits ALK, ROS1, and MET tyrosine kinases, reducing tumor cell proliferation.",
        "Binds to DNA, inhibiting RNA synthesis.",
        "Intercalates into DNA, inhibiting macromolecular biosynthesis.",
        "Intercalates into DNA and inhibits topoisomerase II.",
        "Antagonist of dopamine D₂ and D₃ receptors, enhancing gastrointestinal motility.",
        "Similar to doxorubicin; inhibits topoisomerase II.",
        "Induces ferroptosis by inhibiting the cystine/glutamate antiporter system Xc⁻.",
        "Inhibits the mammalian target of rapamycin (mTOR), affecting cell growth and proliferation.",
        "Induces ferroptosis by promoting degradation of GPX4 and depleting coenzyme Q₁₀.",
        "Inhibits NAMPT, depleting NAD⁺ levels.",
        "Inhibits DNA polymerase and ribonucleotide reductase.",
        "Ubiquitin isopeptidase inhibitor; induces apoptosis.",
        "Inhibits phosphatidylinositol 3-kinase (PI3K), leading to suppression of the PI3K/Akt signaling pathway.",
        "Modulates Bcl-2 family proteins to induce apoptosis.",
        "Inhibits inositol monophosphatase, affecting phosphatidylinositol signaling.",
        "Inhibits HMG-CoA reductase, reducing cholesterol synthesis.",
        "Agonist of peripheral μ-opioid receptors, reducing gastrointestinal motility.",
        "Alkylates DNA, causing cross-linking.",
        "Inhibits microtubule formation, inducing apoptosis.",
        "Inhibits eEF1A to block protein synthesis.",
        "Acts as an ionophore, facilitating the exchange of potassium (K⁺) and hydrogen (H⁺) ions across biological membranes.",
        "Induces apoptosis in mutated KRAS cancer cells.",
        "Forms DNA cross-links, inhibiting DNA replication and transcription.",
        "Antagonist of dopamine receptors, leading to decreased dopamine activity.",
        "Induces apoptosis through mitochondrial pathways and modulates various signaling pathways.",
        "Inhibits HMG-CoA reductase, reducing cholesterol synthesis.",
        "Binds bacterial RNA polymerase, inhibiting RNA synthesis.",
        "Inhibits tumor growth by suppressing the STAT3 signaling pathway and inducing apoptosis.",
        "Inhibits HMG-CoA reductase, reducing cholesterol synthesis.",
        "High affinity and selective activator of Bax.",
        "Inhibits topoisomerase I, causing DNA damage.",
        "Inhibits multiple tyrosine protein kinases, leading to reduced tumor cell proliferation.",
        "Inhibits NF-κB activity and acts as an anti-inflammatory agent.",
        "Inhibits topoisomerase II, leading to DNA strand breaks.",
        "Inhibits topoisomerase I, causing DNA strand breaks.",
        "Positive allosteric modulator of M1 muscarinic acetylcholine receptors, enhancing receptor activity."
    ],
    "Target": [
        "Topoisomerase II", "Mitochondria", "APAF-1", "nAChRs", "Bcl-XL", "Bcl-2", "PARP", "DNA",
        "26S Proteasome", "ATP Synthase", "Topoisomerase I", "DNA", "FASN", "DNA", "DNA Polymerase",
        "ALK, ROS1, MET", "DNA", "Topoisomerase II", "Topoisomerase II", "Dopamine Receptors",
        "Topoisomerase II", "System Xc⁻", "mTOR", "GPX4", "NAMPT", "DNA Polymerase",
        "Ubiquitin Isopeptidase", "PI3K", "Bcl-2", "Inositol Monophosphatase", "HMG-CoA Reductase",
        "μ-Opioid Receptors", "DNA", "TUBB", "eEF1A", "Ion Transport", "KRAS",
        "DNA", "Dopamine Receptors", "Apoptosis Pathways", "HMG-CoA Reductase", "RNA Polymerase",
        "STAT3", "HMG-CoA Reductase", "BAX (Bcl-2)", "Topoisomerase I", "Tyrosine Kinases", "NF-κB",
        "Topoisomerase II", "Topoisomerase I", "M1 Muscarinic Receptors"
    ]
}
table_df = pd.DataFrame(table_data)
table_df["Metadata_cmpdName"] = table_df["Compound"].str.upper()

In [ ]:
def merge_with_obs(adata, compound_df):
    if "Metadata_cmpdName" not in adata.obs:
        raise ValueError("The 'Compound' column must exist in `adata.obs` to perform the merge.")
    # Convert "Compound" in obs to uppercase for merging
    adata.obs["Metadata_cmpdName"] = adata.obs["Metadata_cmpdName"].str.upper()
    # Merge the data
    adata.obs = adata.obs.merge(compound_df, on="Metadata_cmpdName", how="left")
    return adata

In [ ]:
adata_filter_grit = merge_with_obs(adata_filter_grit, table_df)

In [ ]:
adata_filter_grit.obs[["Metadata_cmpdName", "Target", "moa"]].drop_duplicates()

In [ ]:
sc.pl.umap(adata_filter_grit, color ="Target")
sc.pl.umap(adata_filter_grit, color ="moa")
sc.pl.umap(adata_filter_grit, color ="Metadata_cmpdName")